# Baseline (No Masking)

This notebook mirrors the no-masking flow-matching setup from the original baseline.
It is intended as a readable reference implementation for hackathon participants.

**What you will see:**
- Load the pickled dataset
- Pick the most common atom count for fixed-shape batches
- Build a Flow-EGNN conditioned on reactant/product context
- Train briefly with a flow-matching loss
- Integrate an ODE to get a TS structure prediction


## Notebook Outline
1. Imports and device setup
2. Load dataset and inspect keys
3. Choose most common atom count (no masking)
4. Batch builder + flow-matching targets
5. Model definition (Flow-EGNN)
6. Training + evaluation


In [ ]:
# Core imports and runtime device selection for Torch tensors/models.
import pickle
import math
import numpy as np
import torch
import torch.nn as nn

# Prefer Apple Metal (MPS) on Apple Silicon, then CUDA, then CPU fallback.
# This keeps the notebook portable across laptops and GPU servers.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: mps


In [ ]:
# Load the pickled reaction dataset.
# Expected top-level keys include reactant/product/transition_state blocks.
pkl_path = "../Data/transition1x/train.pkl"
with open(pkl_path, "rb") as f:
    obj = pickle.load(f)

# Quick schema sanity checks before downstream tensorization.
print(obj.keys())
print(obj["reactant"].keys())


dict_keys(['reactant', 'transition_state', 'product', 'single_fragment', 'use_ind', 'ts_guess', 'ts_guess_sbv1', 'ts_guess_true', 'ts_guess_NEBCI-xtb'])
dict_keys(['num_atoms', 'charges', 'fragments', 'positions', 'rxn', 'wB97x_6-31G(d).energy', 'wB97x_6-31G(d).atomization_energy', 'wB97x_6-31G(d).forces', 'formula', 'xtb_positions'])


## Fixed Atom Count (No Masking)
ReactOT uses fixed-size tensors. We pick the most common atom count so we can batch without masking.


In [ ]:
# No masking setup: choose one fixed atom count so we can stack arrays directly.
# This avoids padding/masking complexity in the baseline implementation.
Ns = [obj["reactant"]["positions"][i].shape[0] for i in range(2000)]
from collections import Counter
cnt = Counter(Ns)
common_N, common_count = cnt.most_common(1)[0]
print("Most common N:", common_N, "count in first 2000:", common_count)

# Keep only reaction indices that match the chosen atom count.
idxs = [i for i in range(len(obj["reactant"]["positions"]))
        if obj["reactant"]["positions"][i].shape[0] == common_N]
print("Total reactions with that N:", len(idxs))
print("Example indices:", idxs[:10])


Most common N: 10 count in first 2000: 560
Total reactions with that N: 674
Example indices: [122, 123, 124, 125, 126, 127, 128, 129, 130, 131]


In [ ]:
def get_batch(indices, batch_size=16):
    """Return a fixed-size batch of coordinates and atom types.

    Parameters
    ----------
    indices : list[int]
        Candidate reaction indices (already filtered to a common atom count).
    batch_size : int
        Number of reactions to pull from the front of ``indices``.

    Returns
    -------
    xR, xP, xT : torch.Tensor
        Reactant, product, and transition-state coordinates with shape (B, N, 3).
    z : torch.Tensor
        Integer atom identifiers (charges in this dataset) with shape (B, N).
    """
    # Simple deterministic sampling from the provided index list.
    sel = indices[:batch_size]

    # Stack each structure type into a dense numpy batch.
    xR = np.stack([obj["reactant"]["positions"][i] for i in sel], axis=0)
    xP = np.stack([obj["product"]["positions"][i] for i in sel], axis=0)
    xT = np.stack([obj["transition_state"]["positions"][i] for i in sel], axis=0)
    z  = np.stack([np.array(obj["reactant"]["charges"][i], dtype=np.int64) for i in sel], axis=0)

    # Move everything to the configured device and canonical dtypes.
    xR = torch.tensor(xR, dtype=torch.float32, device=device)
    xP = torch.tensor(xP, dtype=torch.float32, device=device)
    xT = torch.tensor(xT, dtype=torch.float32, device=device)
    z  = torch.tensor(z,  dtype=torch.long, device=device)
    return xR, xP, xT, z

xR, xP, xT, z = get_batch(idxs, batch_size=8)
print(xR.shape, xP.shape, xT.shape, z.shape, xR.dtype, z.dtype, xR.device)


torch.Size([8, 10, 3]) torch.Size([8, 10, 3]) torch.Size([8, 10, 3]) torch.Size([8, 10]) torch.float32 torch.int64 mps:0


In [ ]:
def make_flow_batch(xR, xP, xT, z):
    """Create interpolation states and velocity supervision for flow matching.

    We model a path from midpoint ``x0`` to target TS ``x1`` and train the network
    to predict the constant velocity field ``v_star = x1 - x0`` along that path.
    """
    # Midpoint between reactant and product is the baseline starting geometry.
    x0 = 0.5 * (xR + xP)

    # Transition state acts as terminal point for the flow trajectory.
    x1 = xT

    # Sample one time scalar per structure in the batch.
    B  = xR.shape[0]
    t  = torch.rand(B, device=xR.device)

    # Linear interpolation x(t) = (1-t)x0 + t x1.
    x_t = (1 - t)[:, None, None] * x0 + t[:, None, None] * x1

    # Under this linear path, target velocity is constant in t.
    v_star = x1 - x0

    # Conditioning dictionary consumed by the model.
    cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}
    return x_t, t, cond, v_star

x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)
print("x_t", x_t.shape, "t", t.shape, "v*", v_star.shape)
print("t min/max:", float(t.min()), float(t.max()))


x_t torch.Size([8, 10, 3]) t torch.Size([8]) v* torch.Size([8, 10, 3])
t min/max: 0.17483627796173096 0.9171103239059448


## Flow-EGNN Model
The model predicts a velocity field v(x,t) conditioned on reactant/product context and atomic numbers.


In [ ]:
class TimeEmbedding(nn.Module):
    """Deterministic Fourier time embedding for scalar flow times.

    Given ``t`` in [0, 1], produce concatenated sin/cos features at powers of 2
    frequencies so the network can represent multi-scale time dependence.
    """
    def __init__(self, n_freq=16):
        super().__init__()
        # Buffer (not trainable) so it follows module device moves.
        self.register_buffer("freqs", 2 * math.pi * (2 ** torch.arange(n_freq).float()))

    def forward(self, t):
        """Map shape (B,) times to shape (B, 2*n_freq) periodic features."""
        args = t[:, None] * self.freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

te = TimeEmbedding(8).to(device)
out = te(t)
print(out.shape)
print(out[0, :10])


torch.Size([8, 16])
tensor([ 0.9384,  0.6485, -0.9873, -0.3137,  0.5958,  0.9570,  0.5552, -0.9235,
         0.3455, -0.7612], device='mps:0')


In [ ]:
class EGNNLayer(nn.Module):
    """One equivariant GNN block with feature and coordinate updates.

    The layer uses pairwise distance information for message computation,
    aggregates messages by source node, then updates both latent features ``h``
    and coordinates ``x``.
    """
    def __init__(self, h_dim):
        super().__init__()
        # Message network over (h_i, h_j, ||x_i - x_j||^2).
        self.phi_m = nn.Sequential(
            nn.Linear(2*h_dim + 1, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
        )
        # Feature update network fed with aggregated incoming messages.
        self.phi_h = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
        )
        # Scalar edge coefficient used to scale relative coordinate vectors.
        self.phi_x = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 1),
        )

    def forward(self, h, x, src, dst):
        """Run one message-passing step on a precomputed directed edge list."""
        B, N, H = h.shape

        # Gather endpoint coordinates and compute relative geometry.
        xi = x[:, src, :]
        xj = x[:, dst, :]
        rij = xi - xj
        dij = (rij ** 2).sum(dim=-1, keepdim=True)

        # Gather endpoint node features and build edge messages.
        hi = h[:, src, :]
        hj = h[:, dst, :]
        mij = self.phi_m(torch.cat([hi, hj, dij], dim=-1))

        # Aggregate messages back to nodes using scatter-add.
        agg = torch.zeros((B, N, H), device=h.device, dtype=h.dtype)
        agg.scatter_add_(1, src[None, :, None].expand(B, -1, H), mij)

        # Residual feature update.
        h = h + self.phi_h(agg)

        # Coordinate update in relative direction rij scaled by learned scalar.
        s = self.phi_x(mij)
        dx_ij = rij * s
        dx = torch.zeros((B, N, 3), device=x.device, dtype=x.dtype)
        dx.scatter_add_(1, src[None, :, None].expand(B, -1, 3), dx_ij)
        x = x + dx
        return h, x


def fully_connected_edges(N, device):
    """Build a directed fully connected graph (excluding self edges)."""
    idx = torch.arange(N, device=device)
    ii, jj = torch.meshgrid(idx, idx, indexing="ij")

    # Remove diagonal to avoid self-messages.
    mask = ii != jj
    src = ii[mask].reshape(-1)
    dst = jj[mask].reshape(-1)
    return src, dst


In [ ]:
class FlowEGNN(nn.Module):
    """Flow-matching EGNN that predicts per-atom velocity vectors.

    Conditioning combines atom identities with reactant/product geometry context.
    """
    def __init__(self, z_max=100, h_dim=128, n_layers=4, time_freq=16, use_concat_ctx=True):
        super().__init__()
        self.use_concat_ctx = use_concat_ctx

        # Atom-type embedding (reactant charges are used as discrete IDs here).
        self.z_emb = nn.Embedding(z_max+1, h_dim)

        # Time embedding + projection into hidden feature space.
        self.t_emb = TimeEmbedding(n_freq=time_freq)
        self.t_proj = nn.Linear(2*time_freq, h_dim)

        # Optional context: either concatenate (xR, xP, xP-xR) or only displacement.
        if use_concat_ctx:
            self.ctx_proj = nn.Linear(9, h_dim)
        else:
            self.ctx_proj = nn.Linear(3, h_dim)

        # Stack of EGNN interaction layers.
        self.layers = nn.ModuleList([EGNNLayer(h_dim) for _ in range(n_layers)])

        # Final per-node velocity head.
        self.v_head = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 3)
        )

    def forward(self, x_t, t, cond):
        """Predict velocity field ``v(x_t, t | cond)`` for each atom."""
        z = cond["z"]
        xR = cond["xR_ctx"]
        xP = cond["xP_ctx"]

        B, N, _ = x_t.shape

        # Base node features from atom IDs.
        h = self.z_emb(z)

        # Add broadcasted time features to all atoms in each structure.
        te = self.t_proj(self.t_emb(t))
        h = h + te[:, None, :].expand(B, N, -1)

        # Add structural conditioning signal.
        diff = xP - xR
        if self.use_concat_ctx:
            ctx = torch.cat([xR, xP, diff], dim=-1)
        else:
            ctx = diff
        h = h + self.ctx_proj(ctx)

        # Build a dense edge list once per forward pass.
        src, dst = fully_connected_edges(N, x_t.device)

        # Propagate through EGNN stack, updating both h and internal coordinates x.
        x = x_t
        for layer in self.layers:
            h, x = layer(h, x, src, dst)

        # Read out velocity from final node features.
        v = self.v_head(h)
        return v


In [ ]:
# Quick sanity check of model outputs using two context variants:
# 1) displacement-only context; 2) concatenated reactant/product context.
model_diff = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=False).to(device)
model_cat  = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=True ).to(device)

v1 = model_diff(x_t, t, cond)
v2 = model_cat(x_t, t, cond)

print(v1.shape, v2.shape)
print("v1 norm mean:", float(v1.norm(dim=-1).mean()))
print("v2 norm mean:", float(v2.norm(dim=-1).mean()))


torch.Size([8, 10, 3]) torch.Size([8, 10, 3])
v1 norm mean: 0.4055713713169098
v2 norm mean: 0.3963898718357086


/var/folders/8p/6t_fjn5d1c7fxwflz29kdvqr0000gn/T/ipykernel_32006/3374611194.py:8: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("v1 norm mean:", float(v1.norm(dim=-1).mean()))


In [ ]:
def flow_loss(v_pred, v_star):
    """Flow-matching objective: mean squared error on target velocities."""
    return ((v_pred - v_star) ** 2).mean()

# One optimization step to verify gradients and plumbing.
opt = torch.optim.Adam(model_cat.parameters(), lr=1e-3)

opt.zero_grad()
v_pred = model_cat(x_t, t, cond)
loss = flow_loss(v_pred, v_star)
loss.backward()

# Inspect a few gradient norms to catch dead pathways early.
grad_norms = []
for name, p in model_cat.named_parameters():
    if p.grad is not None:
        grad_norms.append((name, float(p.grad.norm().detach().cpu())))
print("loss:", float(loss.detach().cpu()))
print("some grad norms:", grad_norms[:5])


loss: 0.16508516669273376
some grad norms: [('z_emb.weight', 0.027388721704483032), ('t_proj.weight', 0.06450781971216202), ('t_proj.bias', 0.03532533720135689), ('ctx_proj.weight', 0.055339064449071884), ('ctx_proj.bias', 0.03532533720135689)]


## Training
A short training loop to show loss decreasing. This is intentionally small for clarity.


In [ ]:
# Short baseline training loop on randomly sampled interpolation times.
model = model_cat
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

losses = []
for step in range(200):
    # Resample structures each step (still deterministic slice in get_batch).
    xR, xP, xT, z = get_batch(idxs, batch_size=32)

    # Build flow-matching supervision for this batch.
    x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)

    opt.zero_grad()
    v_pred = model(x_t, t, cond)
    loss = ((v_pred - v_star)**2).mean()
    loss.backward()
    opt.step()

    losses.append(float(loss.detach().cpu()))
    if step % 20 == 0:
        print(step, losses[-1])


0 0.13794228434562683
20 0.07306269556283951
40 0.06755601614713669
60 0.06273206323385239
80 0.05780365690588951
100 0.05155413597822189
120 0.043423671275377274
140 0.033506203442811966
160 0.026930542662739754
180 0.021932801231741905


## ODE Integration and RMSD
We integrate the learned velocity field from the midpoint to obtain a TS prediction.


In [ ]:
@torch.no_grad()
def euler_solve(model, x0, cond, steps=50):
    """Integrate ``dx/dt = v(x,t)`` from t=0 to t=1 with explicit Euler."""
    x = x0.clone()
    dt = 1.0 / steps
    for k in range(steps):
        # Use left-endpoint time for each Euler update.
        t = torch.full((x.shape[0],), k * dt, device=x.device, dtype=x.dtype)
        v = model(x, t, cond)
        x = x + dt * v
    return x


def kabsch_align_torch(mobile, reference):
    """
    Kabsch-align a batch of structures to remove global rotation/translation.

    Parameters
    ----------
    mobile : torch.Tensor
        Shape (B, N, 3)
    reference : torch.Tensor
        Shape (B, N, 3)

    Returns
    -------
    mobile_aligned : torch.Tensor
        Kabsch-aligned ``mobile`` coordinates in the centered reference frame.
    reference_centered : torch.Tensor
        Centered reference coordinates used for RMSD computation.
    """
    if mobile.shape != reference.shape:
        raise ValueError("mobile and reference must have the same shape")
    if mobile.ndim != 3 or mobile.shape[-1] != 3:
        raise ValueError("Inputs must have shape (B, N, 3)")

    # Remove batch-wise centroids so only rotational alignment remains.
    mobile_centroid = mobile.mean(dim=1, keepdim=True)      # (B, 1, 3)
    reference_centroid = reference.mean(dim=1, keepdim=True)

    mobile_centered = mobile - mobile_centroid
    reference_centered = reference - reference_centroid

    # Cross-covariance matrix for each structure: (B, 3, 3).
    cov = mobile_centered.transpose(1, 2) @ reference_centered

    # SVD decomposition of covariance.
    U, S, Vh = torch.linalg.svd(cov, full_matrices=False)

    # Enforce a proper rotation (determinant +1) to avoid reflections.
    det = torch.det(Vh.transpose(-2, -1) @ U.transpose(-2, -1))
    corr = torch.eye(3, device=mobile.device, dtype=mobile.dtype).unsqueeze(0).repeat(mobile.shape[0], 1, 1)
    corr[:, -1, -1] = torch.where(det < 0, -1.0, 1.0)

    # Optimal rotation matrix.
    R = Vh.transpose(-2, -1) @ corr @ U.transpose(-2, -1)

    # Rotate centered mobile coordinates into the reference frame.
    mobile_aligned = mobile_centered @ R.transpose(-2, -1)

    return mobile_aligned, reference_centered


def rmsd_torch(A, B):
    """Compute per-structure RMSD for tensors shaped (B, N, 3)."""
    return torch.sqrt(((A - B) ** 2).sum(dim=-1).mean(dim=-1))


def kabsch_rmsd_torch(pred, true):
    """Compute RMSD after rigid alignment of ``pred`` to ``true``."""
    pred_aligned, true_centered = kabsch_align_torch(pred, true)
    return rmsd_torch(pred_aligned, true_centered)


# Example evaluation on a fresh batch.
xR, xP, xT, z = get_batch(idxs, batch_size=16)
x0 = 0.5 * (xR + xP)
cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}

# Integrate predicted flow from midpoint to estimated TS geometry.
xTS_pred = euler_solve(model, x0, cond, steps=80)

# Compare midpoint baseline vs model prediction under aligned RMSD.
r_mid = kabsch_rmsd_torch(x0, xT).mean()
r_pred = kabsch_rmsd_torch(xTS_pred, xT).mean()

print("Kabsch RMSD midpoint->TS:", float(r_mid))
print("Kabsch RMSD pred->TS    :", float(r_pred))


In [ ]:
# Repeat evaluation without Kabsch alignment to include rigid pose differences.
@torch.no_grad()
def euler_solve(model, x0, cond, steps=50):
    """Integrate ``dx/dt = v(x,t)`` with explicit Euler (no alignment step)."""
    x = x0
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((x.shape[0],), k*dt, device=x.device)
        v = model(x, t, cond)
        x = x + dt * v
    return x

xR, xP, xT, z = get_batch(idxs, batch_size=16)
x0 = 0.5 * (xR + xP)
cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}

# Predict TS coordinates by numerical integration.
xTS_pred = euler_solve(model, x0, cond, steps=80)

def rmsd_torch(A, B):
    """Compute per-structure RMSD directly in the current coordinate frame."""
    return torch.sqrt(((A - B)**2).sum(dim=-1).mean(dim=-1))

# Direct RMSD comparison (pose-sensitive).
r_mid = rmsd_torch(x0, xT).mean()
r_pred = rmsd_torch(xTS_pred, xT).mean()
print("RMSD midpoint->TS:", float(r_mid))
print("RMSD pred->TS    :", float(r_pred))


RMSD midpoint->TS: 0.42418259382247925
RMSD pred->TS    : 0.21741563081741333
